# CM4IR baseline on Google Colab
Select **Runtime → Change runtime type → GPU**. This notebook runs the authors' LSUN Cat SR×4 example. Full paper reproduction requires the authors' validation dataset and all checkpoints; see `REPRODUCE.md`. Outputs go to `/content/CM4IR/exp/image_samples`.


In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing'
print('PyTorch', torch.__version__)


In [ ]:
%cd /content`n!git clone https://github.com/tirer-lab/CM4IR.git`n%cd /content/CM4IR`n!git rev-parse HEAD`n!pip install -q einops lpips tqdm pillow pyyaml pandas scipy`n!python -m pip freeze > /content/cm4ir_pip_freeze.txt`n

In [ ]:
from pathlib import Path
import urllib.request
ckpt = Path('/content/CM4IR/exp/logs/lsun_cat/cd_cat256_lpips.pt')
ckpt.parent.mkdir(parents=True, exist_ok=True)
if not ckpt.exists():
    urllib.request.urlretrieve('https://openaipublic.blob.core.windows.net/consistency/cd_cat256_lpips.pt', ckpt)
print('checkpoint bytes:', ckpt.stat().st_size)


Download the **authors' LSUN Cat validation set** from the link in the upstream README. Put images in a class subfolder under `/content/CM4IR/exp/datasets/lsun_cat/` (ImageFolder requires this layout). For persistent storage, mount Drive and copy the dataset there first.


In [ ]:
!pip install -q gdown`n!mkdir -p /content/CM4IR/exp/datasets/lsun_cat/cat`n!gdown --folder --remaining-ok 'https://drive.google.com/drive/folders/1b69dtdPQYjuYnmADzkxSyhPNqjQb6tyN' -O /content/CM4IR/exp/datasets/lsun_cat/cat
from pathlib import Path`nimages = list(Path('/content/CM4IR/exp/datasets/lsun_cat/cat').glob('*.png'))
print('validation images:', len(images))
assert images, 'No LSUN Cat images downloaded'


In [ ]:
%cd /content/CM4IR
!python main.py --config lsun_cat_256.yml --path_y lsun_cat --deg sr_bicubic --deg_scale 4 --sigma_y 0.05 -i CM4IR_lsun_cat_sr_bicubic_sigma_y_0.05 --iN 250 --gamma 0.2 --model_ckpt lsun_cat/cd_cat256_lpips.pt --ni
!find exp/image_samples -maxdepth 2 -type f | head
